# Some Standardized shocks to illustrate the main macroeconomic features of the model

This notebook performs a set of standard macroeconomic shocks on a model and displays results.  It can be used to verify the good functioning of the model and also illustrates nicely how to perform simulations on World Bank models in python using the `modelflow` package.

The simulations performed are:

* 5 separate stimulus shocks equal to 1 percent of *ex ante* GDP.  These can be used to compare fiscal multipliers and the impacts of different kinds of stimulus on GDP, potential GDP, consumption and inflation
    * a 1 percent of GDP decrease in indirect taxes
    * a 1 percent of GDP decrease in direct taxes
    * a 1 percent of GDP increase in government spending on goods and services
    * a 1 percent of GDP increase in government spending on investment goods
    * a 1 percent of GDP increase in government spending on transfers to households
* 4 non-fiscal shocks were also run
    * a temporary one-year 1 percent increase in the monetary policy interest rate;
    * a permanent 10 percent depreciation of the currency
    * a permanent one percent increase in total factor productivity;
    * a permanent \$20 increase in the price of crude oil

**Note:**
This Notebook is designed to be run on any World Bank model. To customize the notebook to run on a different model, the string variable (`Cty`), which is defined in section 1.1 will have to be changed to take the value of the Mnemonic of the country to be simulated. Similarly the location of the file containing the model may have to be revised.   While  the World Bank mnemonics are the same across countries, not all countries report all variables.  As a result for some models, some variable names (notably those of shocked variables or the expenditure variables being held constant) may need to be revised. Otherwise the Notebook should run without change on any World Bank model.

## Set up python environment and load model object

To work with modelflow we must first import the python libraries that we wish to work with and then instantiate the model object, which we have chose to call `themodel`;

In [1]:
from modelclass import model

In [2]:
#Jupyter notebook code that improves the look of the executed notebook
%load_ext autoreload
%autoreload 2

In [3]:
#This is code to manage dependencies if the notebook is executed in the Google Colab cloud service
if 'google.colab' in str(get_ipython()):
  import os
  os.system('apt -qqq install graphviz')
  os.system('pip -qqq install ModelFlowIb   ')


In [4]:
#Set this variable to the three-letter ISO of the country whose model is being simulated
cty="BOL" 
# Models downloaded from the World Bank web site using the model.download_github_repo(reset=1) method of modelflow and
# executing this file from the local version of the file stored on their computer shuold not have to changethe filepath varible
filepath=f'data/{cty}'
themodel,bline = model.modelload(filepath,run=True,keep='Baseline') 

Zipped file read:  data\BOL.pcim


In [5]:
# Replace default definitions / descriptions with more concise versions
custom_description = {
    f'{cty}NECONPRVTXN': "Inflation",
    f'{cty}NYGDPMKTPCN': "Nominal GDP",
    f'{cty}GGEXPTOTLCN': "Government spending",
    f'{cty}GGEXPINTPCN': "Government interest payments",
    f'{cty}GGREVTOTLCN': "Government revenues",
    f'{cty}GGBALOVRLCN': "Fiscal balance",
    f'{cty}GGDBTTOTLCN_': "Public debt (% GDP)" ,      
    f'{cty}NYGDPMKTPKN': "Real GDP",
    f'{cty}NECONPRVTKN': "Real HH Expenditure",
    f'{cty}NYGDPPOTLKN': "Real potenital output",
    f'{cty}NEGDIFTOTKN': "Real investment",
    f'{cty}NEEXPGNFSKN': "Real exports G&S",
    f'{cty}NEIMPGNFSKN': "Real imports G&S",
    f'{cty}GGDBTTOTLCN_': "Public debt (% GDP)"
}


## Prepare the simulations

For each shock, a separate `DataFrame` is created. Each of these `DataFrames` is given a name that evokes the shock to be performed. Then each `DataFrame` is modified to reflect the shock that is to be performed.

Following the creation of the `DataFrames` the shocks will be performed and the results stored using the `keep=` syntax of model flow.

### Fiscal policy shocks

If necessary, the two lines below can be uncommented in order to generate a list of all variables in the model that start GGEXP (general government expenditure) and GGREV (general government revenues) and that end CN (millions of current local currency units).

In [6]:
print('*** Expenditures ***')
themodel['???GGEXP*CN'].des #Uncomment to get list of mnemonics and descriptions for government expenditures
print('')
print('*** Revenues ***')
themodel['???GGREV*CN'].des #Uncomment to get list of mnemonics and descriptions for government revenues

*** Expenditures ***
BOLGGEXPCAPTCN : General Government Capital Expenditure
BOLGGEXPCRNTCN : Current Expenditures
BOLGGEXPGNFSCN : General Government Expenditure on Goods and Services
BOLGGEXPINTDCN : General Government Interest Payments on Domestic Public Debt
BOLGGEXPINTECN : General Government Interest Payments on External Public Debt
BOLGGEXPINTPCN : General Government Interest Payments on Public Debt
BOLGGEXPTOTLCN : General Government Total Expenditure
BOLGGEXPTRNSCN : Current Transfers
BOLGGEXPWAGECN : General Government Wages and Compensation Expenditure

*** Revenues ***
BOLGGREVCOMMCN  : Commodity Related Revenues
BOLGGREVDRCTCN  : General Government Direct Tax Revenues
BOLGGREVGNFSCN  : Taxes on Goods and Services
BOLGGREVGRNTCN  : Grants
BOLGGREVNONTCN  : Non-Tax Revenues
BOLGGREVOTHRCN  : General Government Other Revenues incl. Privatization
BOLGGREVSALECN  : BOLGGREVSALECN
BOLGGREVSOENECN : Renta interna pagada por las empresas publica
BOLGGREVTAXTCN  : Total Tax Revenue

#### Create an expenditures string

The fiscal scenarios below exogenize (hold constant) spending on those elements of government spending that are not being directly shocked.

To facilitate that, the variable `GGexp` is assigned a string containing all of the expenditure variables that are to be held constant.  This variable is then used when setting up each of the fiscal shocks below. 

This list may need to be adjusted from model to model.

In [7]:
# Government spending variables to be held constant
GGexp=f'{cty}GGEXPCAPTCN {cty}GGEXPGNFSCN {cty}GGEXPOTHRCN {cty}GGEXPTOTSCN {cty}GGEXPTRNSCN {cty}GGEXPWAGECN';


### The Indirect tax cut

This shock assumes that the main elements of government spending are held constant at their pre-shock levels.  This assumption could be relaxed by commenting out the second line. 

In the model, indirect taxes are determined as a function of their lagged effective tax rate (Revenues divided by assumed tax base).  

Inspecting the Goods and services tax revenue equation, one can note that the first term drops out in the forecast period (DUMH=0 in the forecast period) and the equation resolves into an identity where revenues evolve according to the tax base and the effective tax rate of the previous year.

To shock the level of indirect spending revenues in 2025 the add factor is used.  By reducing revenues in 2025 by 1 percent of GDP, the effective tax rate in that year falls.  For subsequent year's it is this lower effective rate that is retained, thereby making the shock permanent.

In [8]:
fpol_indirect=bline.copy()
fpol_indirect=themodel.fix(bline,f'{GGexp}') # Freeze other spending levels

# PIT and CIT tax revenues are determined by the lagged effective rate.  
# Thus shocking the level of tax revenues in 2025 by one percent of GDP
# will increase the effective tax rate going forward by a constant amount
# consistent with a .5 percent increase in GDP and the tax base for consumption
fpol_indirect=fpol_indirect.mfcalc(
    f'<2025 2025> {cty}GGREVGNFSCN_A ={cty}GGREVGNFSCN_A - .01*{cty}NYGDPMKTPCN')


#solve the model.
tempdf = themodel(fpol_indirect,silent=1,keep=f'1 % of GDP Indirect tax cut')

The folowing variables are fixed
BOLGGEXPCAPTCN
BOLGGEXPGNFSCN
BOLGGEXPTRNSCN
BOLGGEXPWAGECN


### Direct tax hike of 1% of GDP

The same basic methodology is followed for  direct taxes.


In [9]:
themodel['{cty}GGREVDRCTCN'].frml
themodel['{cty}NECONPRVTKN'].eviews

BOLGGREVDRCTCN : FRML <DAMP,STOC> BOLGGREVDRCTCN = (BOLGGREVDRCTER(-1)/100*BOLNYGDPMKTPCN+61.4227830118073*DREV + BOLGGREVDRCTCN_A)* (1-BOLGGREVDRCTCN_D)+ BOLGGREVDRCTCN_X*BOLGGREVDRCTCN_D $
BOLNECONPRVTKN : 
Not avaible


In [10]:
fpol_direct=bline.copy()
fpol_direct=themodel.fix(bline,f'{GGexp}') # Freeze spending levels Capital

# In the model direct taxes (personal and corporate income taxes) tax are 
# determined by the lagged effective rate of taxation.  Shocking the 
# level of tax revenues in 2025 by one percent of GDP decreases the effective 
# tax rate going forward by a constant amount consistent with a 1 percent decrease 
# in GDP and the tax base for direct taxes (.5% for each)
fpol_direct=fpol_direct.mfcalc(
    f'<2025 2025> {cty}GGREVDRCTCN_A ={cty}GGREVDRCTCN_A  - .01*BOLNYGDPMKTPCN')

#solve the model.
tempdf = themodel(fpol_direct,silent=1,keep=f'1 % of GDP direct tax cut')

The folowing variables are fixed
BOLGGEXPCAPTCN
BOLGGEXPGNFSCN
BOLGGEXPTRNSCN
BOLGGEXPWAGECN


### Increase in expenditure on goods and services

The ex ante fiscal effort is the same in this scenario (1% of ex ante GDP) with the difference that it is implemented as an increase government spending, in this instance on goods and services.



In [11]:
fpol_ExpGS=bline.copy()
fpol_ExpGS=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

fpol_ExpGS=fpol_ExpGS.mfcalc(
    f'<2025 2050> {cty}GGEXPGNFSCN_X ={cty}GGEXPGNFSCN_X   + .01*{cty}NYGDPMKTPCN')


The folowing variables are fixed
BOLGGEXPCAPTCN
BOLGGEXPGNFSCN
BOLGGEXPTRNSCN
BOLGGEXPWAGECN


In [12]:
#solve the model.
tempdf = themodel(fpol_ExpGS,silent=1,keep=f'1 % of GDP increase in G&S spending')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']

### Increase in expenditure on investment goods

The ex ante fiscal effort is the same in this scenario (1% of ex ante GDP), implemented as an increase in government spending on capital goods. 


In [13]:
fpol_ExpInv=bline.copy()
fpol_ExpInv=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

fpol_ExpInv=fpol_ExpInv.mfcalc(f'<2025 2050> {cty}GGEXPCAPTCN_X ={cty}GGEXPCAPTCN_X   + .01*{cty}NYGDPMKTPCN')

The folowing variables are fixed
BOLGGEXPCAPTCN
BOLGGEXPGNFSCN
BOLGGEXPTRNSCN
BOLGGEXPWAGECN


In [14]:
#solve the model.
tempdf = themodel(fpol_ExpInv,silent=1,keep=f'1 % of GDP increase in Govt investment spending')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']



## Comparisons of results from the fiscal scenarios

The following charts compare results from the different fiscal simulations. Impacts will differ both in terms of their long-term and short-term impacts.  For example a scenario that increased investment would likely have negative impacts on consumption in the short-run but in the longer run could be expected to have an opposite impact on potential output, GDP and perhaps consumption.  A scenario that concentrated on transfers or consumption might have more of a short-term impact on demand but in the long run would have limited (and potentially negative impacts on output), especially if increased fiscal deficits and debt crowded out private sector investment.  As all World Bank models are customized to the country for which they have been built the extent of these effects can vary across models.


Recall the scenarios that were run by interrogating the keep_solutions dictionary.



In [15]:
for key,value in themodel.keep_solutions.items():
    print(f"'{key}|'")

'Baseline|'
'1 % of GDP Indirect tax cut|'
'1 % of GDP direct tax cut|'
'1 % of GDP increase in G&S spending|'
'1 % of GDP increase in Govt investment spending|'


### Real GDP impacts and impacts on main Real GDP expenditure components

The following plot uses the `scenarios=` clause to select which scenario results are to be plotted, and then plots them on a series of charts one one page because of the  `samefig=True` and the `datatype=difpctlevel` options.  The first option ensures that all the graphs are arranged in a grid and the second ensures the the results are expressed  as a percent deviation from the first scenario specified in the `scenarios` option -- in this case the baseline scenario. 

Note, the included scenarios in the are identified by the text used in the initial keep command and separated by a horizontal line "|".

In [16]:
fig1=themodel.plot(f'{cty}NYGDPPOTLKN {cty}NYGDPMKTPKN {cty}NECONPRVTKN {cty}NEGDIFTOTKN {cty}NEEXPGNFSKN {cty}NEIMPGNFSKN',
              scenarios='Baseline|'
                   '1 % of GDP Indirect tax cut|'
                   '1 % of GDP direct tax cut|'
                   '1 % of GDP increase in G&S spending|'
                   '1 % of GDP increase in Govt investment spending|'
                   '1 % of GDP increase in transfers to households',
              custom_description=custom_description,
              start=2020, end=2030,datatype='difpctlevel',
              samefig=True,legend=True,show=True,
              title="Comparison of fiscal scenarios",name="FiscalScenarios");

fig1


Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…


For Bolivia, the GDP results appear to be consistent with expectations.  

Fiscal expansion of all types boosted demand and real GDP in the short run (second run).  However, the long run the impact depends on the impact of the spending on potential output (first chart).  

Sustained increased investment spending (red line) served to increase the capital stock and contribute to higher potential and actual GDP.  Spending that focused on consumption (green line) has little discernible impact on potential and therefore GDP impacts tend to subside with time. Potential does rise somewhat in most scenarios, because higher activity induces an initial rise in investment, but this effect is short-lived and dies out and even turn slightly negative toward the end of the projection period, except in the investment and to a lesser extent indirect tax cut scenarios.  

Decreased direct taxation (reduced personal and corporate income taxes) tends to benefit households (orange line), although this impact declines over time due to the relative stability of potential output.  The cut in indirect taxes tends to generate small but persistent benefits (blue line), presumably due to lower prices for domestically produced goods with benefits to both exports and import competing goods.

### Impacts on the fiscal accounts

The following plots shows the impacts on several of the main fiscal indicators. The GDP indicator here is nominal GDP (vs real GDP in the preceding charts).

In [17]:
themodel.plot(f'{cty}NYGDPMKTPCN {cty}GGEXPTOTLCN {cty}GGEXPINTPCN {cty}GGREVTOTLCN {cty}GGBALOVRLCN {cty}GGDBTTOTLCN',
              scenarios='Baseline|1 % of GDP Indirect tax cut|1 % of GDP direct tax cut|'
              '1 % of GDP increase in G&S spending|1 % of GDP increase in Govt investment spending|'
              '1 % of GDP increase in transfers to households',
              custom_description=custom_description,
              start=2020, end=2030,datatype='difpctlevel',samefig=True,legend=True,
              title="Comparison of fiscal scenarios",name="FiscalScenarios")


Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…

Note that:
* Nominal GDP is increased in most scenarios, mainly reflecting the inflationary impact of the scenario (most scenarios saw real GDP decline). 
  *  the exception here is the cut in indirect taxes which reduces consumer prices directly (lower sales taxes) and increases inflationary prices only indirectly by increasing real demand
* The fiscal account deteriorates in all scenarios as compared with the baseline.
    * Nominal spending increases even in the tax scenarios, but here the driver is increased interest payments as other elements of spending were held constant.
    * Although interest payments as a percent of their initial level are up a lot, the increase as a percent of GDP (see next set of charts is less pronounced).
    * Revenues improve in the spending scenarios because of higher nominal GDP.
    * The fiscal balance deteriorates (becomes more negative) with the extent of the deterioration. The decline is smallest in the scenarios where real GDP growth is increasing.
    * Nominal public debt is higher in all scenarios

#### Fiscal impacts as a percent of GDP

As observed, higher inflation (due to increased demand in the early years of the simulation) mean that revenues, expenditures and nominal GDP are all higher in the simulation scenarios.  

The following charts show the results as a percent of GDP.  Here the inflation influences both the numerator and the denominator, so just the net effect is drawn.


In [18]:
# Uncomment to display the mnemonics and desriptions of the general government (GG) expenditure items 
# (EXP) in nominal local currency (CN)
#themodel[f'{cty}GGEXP*CN'].des  

In [19]:
themodel.plot(f'{cty}GGREVTOTLCN {cty}GGEXPTOTLCN {cty}GGEXPINTPCN {cty}GGBALOVRLCN {cty}GGDBTTOTLCN',
              scenarios='Baseline|1 % of GDP Indirect tax cut|1 % of GDP direct tax cut|1 % of GDP increase in G&S spending|1 % of GDP increase in Govt investment spending|1 % of GDP increase in transfers to households',
              start=2023, end=2030,datatype='gdppct',samefig=True,legend=True,title="Comparison of fiscal scenarios",name="FiscalScenarios")

Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…

**Fiscal impacts as percent of GDP**

* Spending as a percent of GDP increases by a bit more than 1 percent of GDP, the result of the original bump up in spending plus increased interest payments as the debt rises.
* Revenues as a percent of GDP are down in the tax reduction scenarios, but up somewhat in the spending scenarios -- presumably reflecting a switch in the mix of total expenditure towards categories that are relatively highly taxed.
* Interest payments rise by about 0.3 percent of GDP by the end of the period due to higher debt levels and because of higher interest rates as debt to GDP rates rise.
* Higher debt and fiscal borrowing will translate into increased competition for domestic and foreign savings, crowding out private sector investment
* The fiscal balance deteriorates, with differences reflecting differences in real GDP and inflation impacts, and revenue impacts.
* Debt rises by more around 5 percent of GDP as the permanent 1 percent increase in spending accumulates over time.

## Non fiscal simulations

Four non-fiscal scenarios were run.  The first a temporary increase in the monetary policy interest rate, the second a 10 percent depreciation, the third is a permanent 1 percent increase in TFP, and the final a permanent $20 increase in the price of crude_petrol.

### Monetary policy shock

In this shock, it is assumed that the central bank raises its policy rate by 1 percentage point for 3 years (2025-27 inclusive).


In [20]:
themodel['*FMLBLLRLCFR'].eviews


BOLFMLBLLRLCFR : Exogen


In [21]:
Mpol=bline.copy()
Mpol=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 


Mpol=Mpol.mfcalc(f'<2025 2027> {cty}FMLBLLRLCFR ={cty}FMLBLLRLCFR   + 1')
#Mpol[f'{cty}FMLBLPOLYFR_X'].loc[2020:2030]/Mpol[f'{cty}FMLBLPOLYFR'].loc[2020:2030]

The folowing variables are fixed
BOLGGEXPCAPTCN
BOLGGEXPGNFSCN
BOLGGEXPTRNSCN
BOLGGEXPWAGECN


In [22]:
#solve the model.
tempdf = themodel(Mpol,silent=1,keep=f'1 ppt increase in policy rate in 2025')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']



In [23]:

themodel.lastdf[f'{cty}FMLBLLRLCFR'].loc[2020:2030]-themodel.basedf[f'{cty}FMLBLLRLCFR'].loc[2020:2030]

2020    0.0
2021    0.0
2022    0.0
2023    0.0
2024    0.0
2025    1.0
2026    1.0
2027    1.0
2028    0.0
2029    0.0
2030    0.0
Name: BOLFMLBLLRLCFR, dtype: float64

### Exchange rate depreciation

This shock assumes a permanent depreciation of the currency by 10 percent in 2025.


In [24]:
themodel['{cty}PANUSATLS'].frml

BOLPANUSATLS : FRML <DAMP,STOC> BOLPANUSATLS = (6.93214810636363+(-6.6969340726087+0.297501474229249*T_LR)*DURING_1986_2008 + BOLPANUSATLS_A)* (1-BOLPANUSATLS_D)+ BOLPANUSATLS_X*BOLPANUSATLS_D $


In [25]:
Mpol_exr=bline.copy()
#Mpol_exr=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

#Shock imposed endogenously (increasing the addfactor by 10 percent of the level of exchange rate )
Mpol_exr=Mpol_exr.mfcalc(f'<2025 2050> {cty}PANUSATLS_A ={cty}PANUSATLS_A + 0.1*{cty}PANUSATLS')


In [26]:
#solve the model.
tempdf = themodel(Mpol_exr,silent=1,keep=f'A permanent 10 percent depreciation in 2025')


### TFP Shock

This shock explores the effect of a permanent increase in the level of TFP by 1 percent beginning in 2025.


In [27]:
TFP=bline.copy()
TFP=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

#TFP=themodel.fix(Mpol,f'{cty}PANUSATLS',2025,2050) # One year shock to MP but then let the equation take over.
TFP=TFP.mfcalc(f'<2025 2050> {cty}NYGDPTFP ={cty}NYGDPTFP * 1.01')


The folowing variables are fixed
BOLGGEXPCAPTCN
BOLGGEXPGNFSCN
BOLGGEXPTRNSCN
BOLGGEXPWAGECN


In [28]:
#solve the model.
tempdf = themodel(TFP,silent=1,keep=f'A permanent 1 percent increase in TFP levels')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']



### A permanent 20 percent increase in oil prices

This shock explores the sensitivity of the model to a permanent 20 USD increase in global oil and natural gas prices beginning in 2025.  The natural gas price are assumed to rise proportionately to a 20 USD increase in world crude oil prices.


In [29]:
themodel['WLDF*GAS*'].des

WLDFNGAS_EUR            : Price of natural gas (USD)
WLDFNGAS_EUR_VALUE_2015 : WLDFNGAS_EUR_VALUE_2015


In [30]:
themodel['WLDF*crude*'].des

WLDFCRUDE_PETRO            : Price of oil (USD)
WLDFCRUDE_PETRO_VALUE_2015 : WLDFCRUDE_PETRO_VALUE_2015


In [31]:
Oil=bline.copy()
Oil=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

# scenario to raise oil prices.  Assume that natural gas prices increase
# by a similar margin as a $20 increase in crude oil prices
Oil=Oil.mfcalc(
    f'<2025 2050> WLDFNHAS_EUR = WLDFNHAS_EUR * (WLDFCRUDE_PETRO+20)/WLDFCRUDE_PETRO')
Oil=Oil.mfcalc(
    f'<2025 2050> WLDFCRUDE_PETRO = WLDFCRUDE_PETRO + 20')
Oil=Oil.mfcalc(
    f'<2025 2050> WLDFCRUDE_PETRO_VALUE_2015 = WLDFCRUDE_PETRO_VALUE_2015 * ( 1 + 20/WLDFCRUDE_PETRO)')


The folowing variables are fixed
BOLGGEXPCAPTCN
BOLGGEXPGNFSCN
BOLGGEXPTRNSCN
BOLGGEXPWAGECN


In [32]:
#solve the model.
tempdf = themodel(Oil,silent=1,keep=f'A permanent $20 increase in global crude oil prices')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']



### Summary  impacts of non-fiscal scenarios

The following graphs show the change in the main macroeconomic indicators (Inflation, proxied here by the inflation rate of household consumption), real GDP, and changes in the current account, fiscal balances and debt levels expressed as a percent of GDP.

In [33]:
for key,value in themodel.keep_solutions.items():
    print(key)

Baseline
1 % of GDP Indirect tax cut
1 % of GDP direct tax cut
1 % of GDP increase in G&S spending
1 % of GDP increase in Govt investment spending
1 ppt increase in policy rate in 2025
A permanent 10 percent depreciation in 2025
A permanent 1 percent increase in TFP levels
A permanent $20 increase in global crude oil prices


In [34]:
scen = (
    "Baseline|"
    "1 ppt increase in policy rate in 2025|"
    "A permanent 10 percent depreciation in 2025|" 
    "A permanent 1 percent increase in TFP levels|"
    "A permanent $20 increase in global crude oil prices"
       )
fig1=themodel.plot(f'{cty}NECONPRVTXN',scenarios=scen,start=2023,end=2030,datatype="difgrowth",
                    heading="Change in inflation",title="Inflation)");
fig2=themodel.plot(f'{cty}NYGDPMKTPKN {cty}NYGDPPOTLKN',scenarios=scen,start=2023,end=2030,datatype="difpctlevel",
                   heading="Percent change from baseline");
fig3=themodel.plot(f'{cty}BNCABFUNDCD {cty}GGBALOVRLCN {cty}GGDBTTOTLCN',scenarios=scen,start=2023,end=2030,datatype="difgdppct",
                   heading="Change as a percent of GDP");
combo=(fig1|fig2|fig3).set_options(samefig=True,name='Non_fisc')
combo.show
                               

<Figure size 2000x1800 with 7 Axes>

As might be expected, effects across these scenarios are more divergent.  

**Inflation**
The exchange rate depreciation is inflationary, reflecting the higher cost of imported goods. Higher domestic inflation induces further depreciation so the inflationary impacts takes some time to exit the system.   The inflation impact in the oil scenario is much smaller than in other countries in large part because Bolivia is both an exporter and importer of crude oil and  domestic prices are administered. 

Both the TFP shock and the monetary policy tightening contribute to declines in inflation.  The TFP shock is deflationary because it raises potential output and therefore opens up a negative output gap, at least initially.  Over time this effect diminishes. The monetary policy shock  also induces an increase in the output gap (due to lower demand) in the short run, but as the economy recovers the inflation rate tends to return to its pre-shock inflation rate.

**Real economy effects**
The permanent increase in TFP translates fairly quickly into higher GDP, and GDP continues to rise -- reflecting stronger potential output.  While the initial increase in TFP will have boosted potential the subsequent additional increases (beyond 1 percent) reflect higher investment induced by the increased productivity of capital (and labor).

The depreciation contributes to a modest increase in GDP as exports and import competing goods benefit from the increased competitiveness. The inflationary effect tends to counter this by eroding the competitiveness advantage initially imparted and by cutting into real incomes. That erosion is apparent in the current account which improves substantially initially but eventually returns to and even below its pre-shock levels. 


Higher oil prices improve the fiscal indicators, while the slower growth from higher interest rates in addition to the increase in borrowing costs impinge on fiscal indicators.  Stronger TFP growth increases real GDP but does not translate into stronger government revenues as a percent of nominal GDP.



#### Real Gdp impacts of non-fiscal simulations

In [35]:

themodel.plot(f'{cty}NYGDPPOTLKN {cty}NYGDPMKTPKN {cty}NECONPRVTKN {cty}NEGDIFTOTKN {cty}NEEXPGNFSKN {cty}NEIMPGNFSKN',
              scenarios=scen,start=2020, end=2030,datatype="difpctlevel",samefig=True,legend=True,
              title="Real GDP impacts of non-fiscal scenarios")

Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…

#### Discussion of real GDP impacts of non-fiscal scenarios

##### The TFP shock (green line)
Potential and real GDP impacts mirror one another. The TFP shock raising potential output permanently and immediately, while real GDP catches up over-time slowly.  The potential GDP impact rises proportionately over time because the higher output induces additional investment which adds further to potential GDP.  Higher potential and actual GDP translates into increased consumption and imports as the economy adjusts to the higher scale of activity.

##### The monetary policy shock (blue line)
As can be expected a tightening of monetary policy has negative effects on GDP. Higher interest rates and slower growth reduce investment growth which has a modest negative effect on potential output and long-run GDP. Consumer demand mirrors GDP as incomes are reduced modestly as compared with the baseline.  Exports are hurt initially due to high capital costs, but as inflation declines they benefit and by the end of the period the impact is negligible.  Imports are similarly hurt initially (as domestic demand weaknes) but recover most of the losses.

##### Depreciation (orange line)
The long run effect of the 10 percent depreciation on potential output is neutral, and in the simulation actual GDP remains elevated compared with the baseline, but over the long-term should be expected to return to an more or less unchanged potential.    Exports benefit initially as domestic goods become more competitive abroad and imports decline as import-competing firms benefit.  However, higher import costs translate into lower real incomes and lower consumption, which persist into the future due to the investment effect on GDP.

##### Oil price hike (red line)

A permanent $20 nominal increase in oil prices has modest impacts on GDP, increasing potential slightly as Bolivia is largely self sufficient in oil.




### Fiscal impacts of non-fiscal scenarios

Fiscal impacts of the non-fiscal scenarios are smaller than those encountered in the fiscal shocks, principally because they reflect only second round effects on government and revenues. 


In [36]:
# Uncomment to display the mnemonics and desriptions of the general gobernment (GG) expenditure items 
# (EXP) in nominal local currency (CN)
themodel[f'{cty}GGEXP*CN'].des  

BOLGGEXPCAPTCN : General Government Capital Expenditure
BOLGGEXPCRNTCN : Current Expenditures
BOLGGEXPGNFSCN : General Government Expenditure on Goods and Services
BOLGGEXPINTDCN : General Government Interest Payments on Domestic Public Debt
BOLGGEXPINTECN : General Government Interest Payments on External Public Debt
BOLGGEXPINTPCN : General Government Interest Payments on Public Debt
BOLGGEXPTOTLCN : General Government Total Expenditure
BOLGGEXPTRNSCN : Current Transfers
BOLGGEXPWAGECN : General Government Wages and Compensation Expenditure


In [37]:
themodel.plot(f'{cty}GGBALOVRLCN {cty}GGEXPTOTLCN {cty}GGREVTOTLCN {cty}GGEXPINTPCN {cty}GGDBTTOTLCN',
              scenarios=scen,start=2020, end=2030,datatype="difgdppct",samefig=True,legend=True,
              title="Revenue impacts of non-fiscal scenarios",name="NonFiscRevenues")

Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…

## Fiscal impacts of the non-fiscal scenarios


The oil price scenario generates a modest improvement in the fiscal balance as expenditures decline (relative to GDP) by more than revenues.

The productivity and depreciation scenarios all see a deterioration in the fiscal balance, reflecting the historically strong income elasticity of government spending.


Government spending is sensitive to the interest rate scenario as higher rates increase the costs of debt servicing. 

Overall the depreciation scenario has the largest impact on the debt to GDP ratio, as the local currency value of foreign debt is increased mechanically by the depreciation, increasing debt as a percent of GDP and increasing the local currency cost of interest paid on foreign debt.
